# ConvNeXt: classificazione di feature reali e sintetiche

## Abstract

Questo notebook addestra un classificatore ConvNeXt compatto per distinguere feature map real, label 0, da fake, label 1. Usa PyTorch per leggere i tensori originali in memory mapping, normalizzarli canale per canale con statistiche calcolate solo sul training e ottimizzare il modello con AdamW, AMP, warm-up, cosine decay, stochastic depth ed early stopping. Il notebook rileva automaticamente ambiente locale o Google Colab, usa Drive come storage persistente in Colab, salva checkpoint riprendibili e produce predizioni e metriche compatibili con il confronto finale.

## Schema della pipeline completa

1. Setup portabile: rileva locale o Colab; in Colab monta Drive e copia i file originali dalla posizione persistente alla VM.
2. Contratti condivisi: verifica sample_index.csv, mapping real/fake, split condiviso e hash stabile.
3. Input e preprocessing: apre i file con memory mapping; recupera un solo campione, lo converte in float32 e applica le stesse statistiche per canale della CNN spatial head.
4. ConvNeXt: proiezione 1280→256, blocchi ConvNeXt a 16×16, downsampling verso 8×8 e 4×4, pooling globale e classificatore binario.
5. Training: batch adattivo, accumulo dei gradienti, AMP soltanto su CUDA, AdamW, warm-up/cosine decay e checkpoint a ogni epoca.
6. Selezione: early stopping e checkpoint migliore scelti soltanto dalla balanced accuracy validation; il test rimane isolato.
7. Consegna: fissa la soglia sul validation, salva metriche, predizioni allineate, figure, TorchScript e checkpoint per la ripresa.

## 1. Setup portabile e cache Colab

Questa cella configura l'unica diramazione locale/Colab. Colab monta Drive, valida la dimensione dei file originali e li porta nella VM temporanea; risultati, checkpoint e dati derivati restano invece nella cartella persistente del progetto.

In [2]:
from pathlib import Path
import importlib.util
import shutil

try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False

EXPECTED_FILES = {"features_0.npy": 2_167_931_008, "features_1.npy": 3_123_445_888}
if IS_COLAB:
    from google.colab import drive
    DRIVE_MOUNT = Path("/content/drive")
    if not (DRIVE_MOUNT / "MyDrive").exists():
        drive.mount(str(DRIVE_MOUNT), force_remount=False)
    PROJECT_ROOT = DRIVE_MOUNT / "MyDrive" / "Magistrale" / "2 Anno" / "Advanced_ML" / "progetto_aml"
    PERSISTENT_DATASET_DIR = PROJECT_ROOT / "dataset"
    if not PERSISTENT_DATASET_DIR.is_dir():
        raise FileNotFoundError(f"Dataset directory non trovata su Drive: {PERSISTENT_DATASET_DIR}")
    RUNTIME_PROJECT_ROOT = Path("/content/progetto_aml")
    RUNTIME_DATASET_DIR = RUNTIME_PROJECT_ROOT / "dataset"
    RUNTIME_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    for filename, expected_size in EXPECTED_FILES.items():
        source, destination = PERSISTENT_DATASET_DIR / filename, RUNTIME_DATASET_DIR / filename
        if not source.is_file() or source.stat().st_size != expected_size:
            raise RuntimeError(f"File originale assente o non valido: {source}")
        if not destination.is_file() or destination.stat().st_size != expected_size:
            print(f"Copio {filename} da Drive alla VM Colab...")
            shutil.copy2(source, destination)
        if destination.stat().st_size != expected_size:
            raise RuntimeError(f"Copia incompleta: {destination}")
    DATASET_DIR = RUNTIME_DATASET_DIR
    PROCESSED_DATASET_DIR = PERSISTENT_DATASET_DIR
    ARTIFACTS_DIR, RESULTS_DIR = PROJECT_ROOT / "artifacts", PROJECT_ROOT / "results"
else:
    PROJECT_ROOT = Path.cwd().resolve()
    if not (PROJECT_ROOT / "dataset").is_dir():
        PROJECT_ROOT = PROJECT_ROOT.parent
    RUNTIME_PROJECT_ROOT = PROJECT_ROOT
    PERSISTENT_DATASET_DIR = DATASET_DIR = PROCESSED_DATASET_DIR = PROJECT_ROOT / "dataset"
    ARTIFACTS_DIR, RESULTS_DIR = PROJECT_ROOT / "artifacts", PROJECT_ROOT / "results"

if not PERSISTENT_DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Dataset directory non trovata: {PERSISTENT_DATASET_DIR}")
for directory in (PROCESSED_DATASET_DIR, ARTIFACTS_DIR, RESULTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print("Ambiente:", "Google Colab" if IS_COLAB else "Locale")
print("Dataset per lettura:", DATASET_DIR)
print("Risultati persistenti:", RESULTS_DIR)

Copio features_0.npy da Drive alla VM Colab...
Copio features_1.npy da Drive alla VM Colab...
Ambiente: Google Colab
Dataset per lettura: /content/progetto_aml/dataset
Risultati persistenti: /content/drive/MyDrive/Magistrale/2 Anno/Advanced_ML/progetto_aml/results


## 2. Dipendenze, riproducibilità e configurazione hardware

Questa cella installa soltanto pacchetti assenti, registra le versioni, imposta i semi e rileva GPU e memoria. La dimensione del batch è adattata alla VRAM e l'accumulo dei gradienti mantiene un batch effettivo di circa 32 esempi.

In [3]:
import hashlib
import importlib
import json
import math
import platform
import random
import subprocess
import sys
import time
from contextlib import nullcontext
from datetime import datetime, timezone

required = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "sklearn": "scikit-learn", "torch": "torch", "tqdm": "tqdm"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
from sklearn.calibration import calibration_curve
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, brier_score_loss, confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED, MAX_EPOCHS, PATIENCE = 42, 100, 15
LEARNING_RATE, WEIGHT_DECAY, WARMUP_EPOCHS = 3e-4, 1e-2, 5
EFFECTIVE_BATCH_SIZE, GRADIENT_CLIP_NORM, DROPOUT, EPSILON = 32, 1.0, 0.30, 1e-6
MODEL_NAME, RESUME_RUN_ID = "convnext", None
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"
if AMP_ENABLED:
    gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
    BATCH_SIZE = 16 if gpu_memory_gib >= 12 else 8 if gpu_memory_gib >= 6 else 4
    GPU_NAME = torch.cuda.get_device_name(0)
else:
    gpu_memory_gib, BATCH_SIZE, GPU_NAME = 0.0, 2, None
NUM_WORKERS = 2 if IS_COLAB else 0
ACCUMULATION_STEPS = math.ceil(EFFECTIVE_BATCH_SIZE / BATCH_SIZE)
RUN_ID = RESUME_RUN_ID or f"seed{SEED}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn.__version__, "torch": torch.__version__, "cuda": torch.version.cuda, "device": str(DEVICE), "gpu_name": GPU_NAME, "gpu_memory_gib": gpu_memory_gib}
print(json.dumps(VERSIONS, indent=2))
print(f"Batch per device: {BATCH_SIZE}; accumulo: {ACCUMULATION_STEPS}; batch effettivo: {BATCH_SIZE * ACCUMULATION_STEPS}")

{
  "python": "3.13.15",
  "numpy": "2.1.3",
  "pandas": "2.2.3",
  "scikit_learn": "1.6.1",
  "torch": "2.11.0+cpu",
  "cuda": null,
  "device": "cpu",
  "gpu_name": null,
  "gpu_memory_gib": 0.0
}
Batch per device: 2; accumulo: 16; batch effettivo: 32


## 3. Indice dei campioni e split comune

Questa cella dichiara il mapping delle classi e verifica l'indice persistente. Carica lo split comune, oppure lo crea in modo stratificato se manca, quindi calcola l'hash con la medesima serializzazione usata dalle statistiche spaziali e dagli altri modelli.

In [5]:
CLASS_MAPPING = {"features_0.npy": {"label": 0, "name": "real"}, "features_1.npy": {"label": 1, "name": "fake"}}
POSITIVE_LABEL, POSITIVE_CLASS, N_REAL, N_FAKE = 1, "fake", 1654, 2383
N_SAMPLES = N_REAL + N_FAKE
SAMPLE_INDEX_PATH, SPLIT_PATH = PROCESSED_DATASET_DIR / "sample_index.csv", ARTIFACTS_DIR / "splits_seed42.npz"
if not SAMPLE_INDEX_PATH.is_file():
    raise FileNotFoundError(f"Indice campioni mancante: {SAMPLE_INDEX_PATH}. Eseguire prima dataset_exploration.ipynb.")
sample_index = pd.read_csv(SAMPLE_INDEX_PATH)
required_columns = {"sample_id", "source_file", "source_index", "label"}
if len(sample_index) != N_SAMPLES or not required_columns.issubset(sample_index.columns) or not sample_index["sample_id"].is_unique:
    raise ValueError("sample_index.csv non rispetta il contratto comune")
if not set(sample_index["source_file"]).issubset(CLASS_MAPPING):
    raise ValueError("source_file contiene valori inattesi")
labels = sample_index["label"].to_numpy(dtype=np.int64)
if not np.array_equal(labels, sample_index["source_file"].map(lambda name: CLASS_MAPPING[name]["label"]).to_numpy()):
    raise ValueError("Le label non rispettano il mapping real/fake dichiarato")

if SPLIT_PATH.is_file():
    split_file = np.load(SPLIT_PATH, allow_pickle=False)
    train_indices, validation_indices, test_indices = (split_file[name].astype(np.int64) for name in ("train_indices", "validation_indices", "test_indices"))
else:
    indices = np.arange(N_SAMPLES, dtype=np.int64)
    train_indices, held_out = train_test_split(indices, test_size=0.30, stratify=labels, random_state=SEED)
    validation_indices, test_indices = train_test_split(held_out, test_size=0.50, stratify=labels[held_out], random_state=SEED)
    np.savez_compressed(SPLIT_PATH, train_indices=np.sort(train_indices), validation_indices=np.sort(validation_indices), test_indices=np.sort(test_indices))
train_indices, validation_indices, test_indices = map(np.sort, (train_indices, validation_indices, test_indices))
merged = np.concatenate([train_indices, validation_indices, test_indices])
if len(np.unique(merged)) != N_SAMPLES or not np.array_equal(np.sort(merged), np.arange(N_SAMPLES)):
    raise ValueError("Split non disgiunto, incompleto o fuori range")
digest = hashlib.sha256()
for split_name, indices in (("train", train_indices), ("validation", validation_indices), ("test", test_indices)):
    digest.update(split_name.encode("utf-8"))
    digest.update(np.asarray(indices, dtype="<i8").tobytes())
SPLIT_HASH = digest.hexdigest()
for split_name, indices in (("train", train_indices), ("validation", validation_indices), ("test", test_indices)):
    counts = np.bincount(labels[indices], minlength=2)
    print(f"{split_name}: {len(indices)} campioni, real={counts[0]}, fake={counts[1]}")
print("Split hash:", SPLIT_HASH)

train: 2825 campioni, real=1157, fake=1668
validation: 606 campioni, real=248, fake=358
test: 606 campioni, real=249, fake=357
Split hash: e23bd01e9ec079e665ed5cb58cdba6c190d40bb702919412e2f5f5aeca63cbd6


## 4. Memory mapping e normalizzazione condivisa

Questa cella apre le copie locali dei tensori in modalità memory-mapped e valida shape e dtype. Carica poi le statistiche per canale costruite sul solo training e verifica l'hash: il preprocessing è quindi identico a quello della CNN spatial head.

In [10]:
features_real = np.load(DATASET_DIR / "features_0.npy", mmap_mode="r", allow_pickle=False)
features_fake = np.load(DATASET_DIR / "features_1.npy", mmap_mode="r", allow_pickle=False)
if features_real.shape != (N_REAL, 1280, 16, 16) or features_fake.shape != (N_FAKE, 1280, 16, 16):
    raise ValueError("Shape delle feature map non valida")
if features_real.dtype != np.float32 or features_fake.dtype != np.float32:
    raise TypeError("Le feature map devono essere float32")

normalization_path = PROCESSED_DATASET_DIR / "spatial_channel_normalization.npz"
if not normalization_path.is_file():
    raise FileNotFoundError(f"Statistiche spaziali mancanti: {normalization_path}")
normalization = np.load(normalization_path, allow_pickle=False)
channel_mean = np.asarray(normalization["channel_mean"], dtype=np.float32)
channel_std = np.asarray(normalization["channel_std"], dtype=np.float32)
if channel_mean.shape != (1280,) or channel_std.shape != (1280,) or not np.isfinite(channel_mean).all() or not np.isfinite(channel_std).all() or np.any(channel_std <= 0):
    raise ValueError("Statistiche di normalizzazione non valide")
if not np.isclose(float(normalization["epsilon"]), EPSILON) or str(normalization["split_hash"]) != SPLIT_HASH:
    raise ValueError("Normalizzazione e split non coincidono")
print("Feature real:", features_real.shape, features_real.dtype)
print("Feature fake:", features_fake.shape, features_fake.dtype)
print("Campioni nel training delle statistiche:", int(normalization["training_samples"]))

Feature real: (1654, 1280, 16, 16) float32
Feature fake: (2383, 1280, 16, 16) float32
Campioni nel training delle statistiche: 2825


## 5. Dataset senza augmentation e architettura ConvNeXt

Questa cella implementa il Dataset deterministico: accede al file stabilito da sample_index.csv, copia solo il campione richiesto in float32 e normalizza i canali senza modifiche casuali. Definisce un ConvNeXt da circa 8 milioni di parametri con stochastic depth interna e dropout nel classificatore.

### Struttura del ConvNeXt implementato

Input normalizzato: 1280 × 16 × 16

→ Stem: Conv 1×1, 1280 → 256 + LayerNorm2d

→ Stage 1: 3 ConvNeXt block a 256 canali, risoluzione 16 × 16

→ Downsample: LayerNorm2d + Conv 2×2 stride 2, 256 → 384, risoluzione 8 × 8

→ Stage 2: 3 ConvNeXt block a 384 canali, risoluzione 8 × 8

→ Downsample: LayerNorm2d + Conv 2×2 stride 2, 384 → 512, risoluzione 4 × 4

→ Stage 3: 2 ConvNeXt block a 512 canali, risoluzione 4 × 4

→ LayerNorm2d → global average pooling → Dropout 0.30 → Linear 512 → 1 logit

Ogni ConvNeXt block usa convoluzione depthwise 7×7, LayerNorm, MLP con espansione 3×, GELU, layer scaling e stochastic depth progressiva fino a 0.10. Il modello contiene circa 8.66 milioni di parametri addestrabili.

In [11]:
class SpatialFeatureDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices, dtype=np.int64)
        self.source_files = sample_index["source_file"].to_numpy()
        self.source_indices = sample_index["source_index"].to_numpy(dtype=np.int64)
        self.targets = labels.astype(np.float32)
        self.mean = channel_mean.reshape(1280, 1, 1)
        self.std = np.maximum(channel_std.reshape(1280, 1, 1), np.float32(EPSILON))
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, position):
        global_index = int(self.indices[position])
        source, local_index = self.source_files[global_index], int(self.source_indices[global_index])
        if source == "features_0.npy":
            values = features_real[local_index]
        elif source == "features_1.npy":
            values = features_fake[local_index]
        else:
            raise ValueError(f"Source file non supportato: {source}")
        values = np.array(values, dtype=np.float32, copy=True)
        values -= self.mean
        values /= self.std
        return torch.from_numpy(values), torch.tensor(self.targets[global_index], dtype=torch.float32)

class LayerNorm2d(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.LayerNorm(channels)
    def forward(self, inputs):
        return self.norm(inputs.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)

class DropPath(nn.Module):
    def __init__(self, probability=0.0):
        super().__init__()
        self.probability = float(probability)
    def forward(self, inputs):
        if self.probability == 0.0 or not self.training:
            return inputs
        keep_probability = 1.0 - self.probability
        mask = keep_probability + torch.rand((inputs.shape[0], 1, 1, 1), dtype=inputs.dtype, device=inputs.device)
        mask.floor_()
        return inputs.div(keep_probability) * mask

class ConvNeXtBlock(nn.Module):
    def __init__(self, channels, drop_path=0.0, mlp_ratio=3):
        super().__init__()
        hidden = channels * mlp_ratio
        self.depthwise = nn.Conv2d(channels, channels, kernel_size=7, padding=3, groups=channels)
        self.norm = nn.LayerNorm(channels)
        self.expand = nn.Linear(channels, hidden)
        self.activation = nn.GELU()
        self.project = nn.Linear(hidden, channels)
        self.gamma = nn.Parameter(1e-6 * torch.ones(channels))
        self.drop_path = DropPath(drop_path)
    def forward(self, inputs):
        residual = inputs
        outputs = self.depthwise(inputs).permute(0, 2, 3, 1)
        outputs = self.norm(outputs)
        outputs = self.project(self.activation(self.expand(outputs)))
        outputs = (self.gamma * outputs).permute(0, 3, 1, 2)
        return residual + self.drop_path(outputs)

class ConvNeXtFeatureClassifier(nn.Module):
    def __init__(self, dropout=DROPOUT, stochastic_depth=0.10):
        super().__init__()
        rates = torch.linspace(0, stochastic_depth, steps=8).tolist()
        self.stem = nn.Sequential(nn.Conv2d(1280, 256, kernel_size=1, bias=False), LayerNorm2d(256))
        self.stage_one = nn.Sequential(*[ConvNeXtBlock(256, rates[index]) for index in range(3)])
        self.down_one = nn.Sequential(LayerNorm2d(256), nn.Conv2d(256, 384, kernel_size=2, stride=2))
        self.stage_two = nn.Sequential(*[ConvNeXtBlock(384, rates[index + 3]) for index in range(3)])
        self.down_two = nn.Sequential(LayerNorm2d(384), nn.Conv2d(384, 512, kernel_size=2, stride=2))
        self.stage_three = nn.Sequential(*[ConvNeXtBlock(512, rates[index + 6]) for index in range(2)])
        self.final_norm = LayerNorm2d(512)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(1), nn.Dropout(dropout), nn.Linear(512, 1))
    def forward(self, inputs):
        outputs = self.stage_one(self.stem(inputs))
        outputs = self.stage_two(self.down_one(outputs))
        outputs = self.stage_three(self.down_two(outputs))
        return self.classifier(self.pool(self.final_norm(outputs))).squeeze(1)

train_dataset, validation_dataset, test_dataset = (SpatialFeatureDataset(indices) for indices in (train_indices, validation_indices, test_indices))
loader_options = {"batch_size": BATCH_SIZE, "num_workers": NUM_WORKERS, "pin_memory": AMP_ENABLED}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_options)
validation_loader = DataLoader(validation_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)
model = ConvNeXtFeatureClassifier().to(DEVICE)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Parametri addestrabili: {parameter_count:,}")

Parametri addestrabili: 8,658,945


## 6. Directory persistente, checkpoint e metriche

Questa cella prepara la directory dell'esecuzione. Una nuova esecuzione riceve un run_id temporale e non sovrascrive risultati; impostando RESUME_RUN_ID nella configurazione si riapre invece il checkpoint persistente. Include inferenza e metriche comuni con fake come classe positiva.

In [12]:
RUN_DIR, FIGURES_DIR = RESULTS_DIR / MODEL_NAME / RUN_ID, RESULTS_DIR / MODEL_NAME / RUN_ID / "figures"
if RESUME_RUN_ID is None:
    RUN_DIR.mkdir(parents=True, exist_ok=False)
else:
    if not RUN_DIR.is_dir() or not (RUN_DIR / "last_checkpoint.pt").is_file():
        raise FileNotFoundError(f"Checkpoint per la ripresa non trovato: {RUN_DIR}")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def save_checkpoint(payload, filename):
    final_path = RUN_DIR / filename
    if IS_COLAB:
        temporary_path = RUNTIME_PROJECT_ROOT / filename
        torch.save(payload, temporary_path)
        shutil.copy2(temporary_path, final_path)
        temporary_path.unlink()
    else:
        torch.save(payload, final_path)
    if not final_path.is_file():
        raise RuntimeError(f"Checkpoint non salvato: {final_path}")

def autocast_context():
    return torch.autocast(device_type="cuda", dtype=torch.float16) if AMP_ENABLED else nullcontext()

@torch.no_grad()
def predict(loader, network=model):
    network.eval()
    logits_parts, target_parts = [], []
    for values, targets in loader:
        values = values.to(DEVICE, non_blocking=AMP_ENABLED)
        with autocast_context():
            logits_parts.append(network(values).float().cpu().numpy())
        target_parts.append(targets.numpy())
    logits = np.concatenate(logits_parts).astype(np.float64)
    targets = np.concatenate(target_parts).astype(np.int64)
    return targets, logits, 1.0 / (1.0 + np.exp(-logits))

def logits_loss(logits, targets):
    return float(nn.functional.binary_cross_entropy_with_logits(torch.tensor(logits, dtype=torch.float32), torch.tensor(targets, dtype=torch.float32)).item())

def calculate_metrics(targets, logits, probabilities, threshold, loss=np.nan):
    predicted = (probabilities >= threshold).astype(np.int64)
    matrix = confusion_matrix(targets, predicted, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()
    try:
        roc_auc, pr_auc, brier = roc_auc_score(targets, probabilities), average_precision_score(targets, probabilities), brier_score_loss(targets, probabilities)
    except ValueError:
        roc_auc = pr_auc = brier = np.nan
    return {"loss": float(loss), "accuracy": accuracy_score(targets, predicted), "balanced_accuracy": balanced_accuracy_score(targets, predicted), "precision": precision_score(targets, predicted, pos_label=1, zero_division=0), "recall": recall_score(targets, predicted, pos_label=1, zero_division=0), "specificity": tn / (tn + fp) if tn + fp else np.nan, "f1": f1_score(targets, predicted, pos_label=1, zero_division=0), "roc_auc": roc_auc, "pr_auc": pr_auc, "brier_score": brier, "confusion_matrix": matrix}

## 7. Training con warm-up, cosine decay e ripresa

Questa cella addestra sul solo training con accumulo dei gradienti. Il learning rate usa warm-up e cosine decay; a ogni epoca salva last_checkpoint.pt e history.csv, permettendo di riprendere un'interruzione. Il checkpoint migliore è scelto esclusivamente sulla balanced accuracy validation.

In [13]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
updates_per_epoch = math.ceil(len(train_loader) / ACCUMULATION_STEPS)
total_updates = MAX_EPOCHS * updates_per_epoch
warmup_updates = max(1, WARMUP_EPOCHS * updates_per_epoch)
def learning_rate_multiplier(step):
    if step < warmup_updates:
        return (step + 1) / warmup_updates
    progress = (step - warmup_updates) / max(1, total_updates - warmup_updates)
    return 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=learning_rate_multiplier)
criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
history, start_epoch, best_score, best_epoch, stale_epochs = [], 1, -np.inf, 0, 0

if RESUME_RUN_ID is not None:
    resume = torch.load(RUN_DIR / "last_checkpoint.pt", map_location=DEVICE, weights_only=False)
    if resume["split_hash"] != SPLIT_HASH:
        raise RuntimeError("Il checkpoint da riprendere appartiene a uno split differente")
    model.load_state_dict(resume["model_state_dict"])
    optimizer.load_state_dict(resume["optimizer_state_dict"])
    scheduler.load_state_dict(resume["scheduler_state_dict"])
    scaler.load_state_dict(resume["scaler_state_dict"])
    start_epoch, best_score, best_epoch, stale_epochs = resume["epoch"] + 1, resume["best_score"], resume["best_epoch"], resume["stale_epochs"]
    history_path = RUN_DIR / "history.csv"
    history = pd.read_csv(history_path).to_dict("records") if history_path.is_file() else []
    print(f"Ripresa dall'epoca {start_epoch}; migliore balanced accuracy: {best_score:.4f}")

training_started = time.perf_counter()
for epoch in range(start_epoch, MAX_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    for batch_number, (values, targets) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch:03d}", leave=False), start=1):
        values, targets = values.to(DEVICE, non_blocking=AMP_ENABLED), targets.to(DEVICE, non_blocking=AMP_ENABLED)
        with autocast_context():
            loss = criterion(model(values), targets) / ACCUMULATION_STEPS
        scaler.scale(loss).backward()
        if batch_number % ACCUMULATION_STEPS == 0 or batch_number == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

    train_y, train_logits, train_p = predict(train_loader)
    val_y, val_logits, val_p = predict(validation_loader)
    train_loss, val_loss = logits_loss(train_logits, train_y), logits_loss(val_logits, val_y)
    train_metrics, val_metrics = calculate_metrics(train_y, train_logits, train_p, 0.5, train_loss), calculate_metrics(val_y, val_logits, val_p, 0.5, val_loss)
    history.append({"step": epoch, "train_loss": train_loss, "val_loss": val_loss, "train_accuracy": train_metrics["accuracy"], "val_accuracy": val_metrics["accuracy"], "learning_rate": optimizer.param_groups[0]["lr"], "val_balanced_accuracy": val_metrics["balanced_accuracy"]})
    improved = val_metrics["balanced_accuracy"] > best_score + 1e-12
    if improved:
        best_score, best_epoch, stale_epochs = val_metrics["balanced_accuracy"], epoch, 0
        save_checkpoint({"epoch": epoch, "model_state_dict": model.state_dict(), "validation_balanced_accuracy": best_score, "split_hash": SPLIT_HASH}, "best_state_dict.pt")
    else:
        stale_epochs += 1
    history_df = pd.DataFrame(history)
    history_df.to_csv(RUN_DIR / "history.csv", index=False)
    save_checkpoint({"epoch": epoch, "model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(), "scheduler_state_dict": scheduler.state_dict(), "scaler_state_dict": scaler.state_dict(), "best_score": best_score, "best_epoch": best_epoch, "stale_epochs": stale_epochs, "split_hash": SPLIT_HASH}, "last_checkpoint.pt")
    print(f"Epoch {epoch:03d}: validation balanced accuracy={val_metrics['balanced_accuracy']:.4f}; lr={optimizer.param_groups[0]['lr']:.2e}")
    if stale_epochs >= PATIENCE:
        print("Early stopping; miglior epoca:", best_epoch)
        break

training_time_seconds = time.perf_counter() - training_started
best_checkpoint = torch.load(RUN_DIR / "best_state_dict.pt", map_location=DEVICE, weights_only=False)
if best_checkpoint["split_hash"] != SPLIT_HASH:
    raise RuntimeError("Il miglior checkpoint appartiene a uno split differente")
model.load_state_dict(best_checkpoint["model_state_dict"])
history_df = pd.DataFrame(history)
print(f"Tempo di training di questa sessione: {training_time_seconds:.1f} s")

Epoch 001:   0%|          | 0/1413 [00:00<?, ?it/s]

Epoch 001: validation balanced accuracy=0.9367; lr=6.07e-05


Epoch 002:   0%|          | 0/1413 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 8. Soglia validation, metriche e predizioni allineate

Questa cella seleziona una soglia soltanto dalle probabilità validation del checkpoint migliore. Solo dopo questa selezione valuta train, validation e test, salvando logit e probabilità per sample_id nel formato richiesto dall'ensemble.

In [ ]:
val_y, val_logits, val_p = predict(validation_loader)
threshold_candidates = np.unique(np.concatenate([[0.0], val_p, [0.5, 1.0]]))
threshold = float(threshold_candidates[np.argmax([balanced_accuracy_score(val_y, val_p >= candidate) for candidate in threshold_candidates])])
print("Soglia selezionata sul validation:", threshold)

prediction_frames, metric_rows = [], []
for split_name, indices, loader in (("train", train_indices, train_loader), ("validation", validation_indices, validation_loader), ("test", test_indices, test_loader)):
    inference_started = time.perf_counter()
    targets, logits, probabilities = predict(loader)
    inference_time = time.perf_counter() - inference_started
    values = calculate_metrics(targets, logits, probabilities, threshold, logits_loss(logits, targets))
    metric_rows.append({"run_id": RUN_ID, "model": MODEL_NAME, "split": split_name, "n_samples": len(targets), **{key: value for key, value in values.items() if key != "confusion_matrix"}, "threshold": threshold, "training_time_seconds": training_time_seconds, "inference_time_seconds": inference_time})
    frame = sample_index.iloc[indices][["sample_id", "source_file", "source_index"]].copy()
    frame["split"], frame["y_true"], frame["raw_score"], frame["probability"] = split_name, targets, logits, probabilities
    frame["y_pred"] = (probabilities >= threshold).astype(np.int64)
    prediction_frames.append(frame)

metrics_df, predictions_df = pd.DataFrame(metric_rows), pd.concat(prediction_frames, ignore_index=True)
if len(predictions_df) != N_SAMPLES or not predictions_df["sample_id"].is_unique:
    raise RuntimeError("Predizioni non allineate una-a-una ai sample_id")
metrics_df.to_csv(RUN_DIR / "metrics.csv", index=False)
predictions_df.to_csv(RUN_DIR / "predictions.csv", index=False)
display(metrics_df.round(4))

## 9. Figure di training e valutazione

Questa cella salva figure adatte alla relazione: curve di loss e accuracy, matrice di confusione per split, ROC, precision-recall e calibrazione. I grafici sono salvati a 200 DPI prima della visualizzazione.

In [ ]:
def save_figure(filename):
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()

plt.figure(figsize=(8, 4))
plt.plot(history_df["step"], history_df["train_loss"], label="Train")
plt.plot(history_df["step"], history_df["val_loss"], label="Validation")
plt.title("ConvNeXt — loss"); plt.xlabel("Epoca"); plt.ylabel("BCE loss"); plt.grid(alpha=0.3); plt.legend()
save_figure("loss_curve.png")

plt.figure(figsize=(8, 4))
plt.plot(history_df["step"], history_df["train_accuracy"], label="Train")
plt.plot(history_df["step"], history_df["val_accuracy"], label="Validation")
plt.title("ConvNeXt — accuracy"); plt.xlabel("Epoca"); plt.ylabel("Accuracy"); plt.grid(alpha=0.3); plt.legend()
save_figure("accuracy_curve.png")

for split_name in ("train", "validation", "test"):
    frame = predictions_df.loc[predictions_df["split"] == split_name]
    matrix = confusion_matrix(frame["y_true"], frame["y_pred"], labels=[0, 1])
    plt.figure(figsize=(5, 4)); plt.imshow(matrix, cmap="Blues")
    plt.title(f"Matrice di confusione — {split_name}"); plt.xticks([0, 1], ["real", "fake"]); plt.yticks([0, 1], ["real", "fake"])
    plt.xlabel("Predetto"); plt.ylabel("Reale")
    for row in range(2):
        for column in range(2):
            plt.text(column, row, str(matrix[row, column]), ha="center", va="center")
    plt.colorbar()
    save_figure(f"confusion_matrix_{split_name}.png")

test_frame = predictions_df.loc[predictions_df["split"] == "test"]
test_y, test_p = test_frame["y_true"].to_numpy(), test_frame["probability"].to_numpy()
fpr, tpr, _ = roc_curve(test_y, test_p)
plt.figure(figsize=(6, 5)); plt.plot(fpr, tpr, label=f"AUC={roc_auc_score(test_y, test_p):.3f}"); plt.plot([0, 1], [0, 1], "--", color="gray")
plt.title("ROC sul test"); plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.grid(alpha=0.3); plt.legend()
save_figure("roc_curve.png")

precision, recall, _ = precision_recall_curve(test_y, test_p)
plt.figure(figsize=(6, 5)); plt.plot(recall, precision, label=f"AP={average_precision_score(test_y, test_p):.3f}")
plt.title("Precision-recall sul test"); plt.xlabel("Recall fake"); plt.ylabel("Precision fake"); plt.grid(alpha=0.3); plt.legend()
save_figure("precision_recall_curve.png")

observed, predicted = calibration_curve(test_y, test_p, n_bins=10, strategy="quantile")
plt.figure(figsize=(6, 5)); plt.plot(predicted, observed, marker="o", label="ConvNeXt"); plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfetta")
plt.title("Calibrazione sul test"); plt.xlabel("Probabilità predetta"); plt.ylabel("Frazione fake osservata"); plt.grid(alpha=0.3); plt.legend()
save_figure("calibration_curve.png")

## 10. Esportazione TorchScript e verifica di ricarica

Questa cella salva la configurazione riproducibile, esporta il migliore modello in TorchScript e ricarica sia pesi sia TorchScript. Confronta i logit su un batch validation per verificare che l'artefatto portabile sia coerente con il checkpoint.

In [ ]:
model_config = {
    "model_name": MODEL_NAME, "run_id": RUN_ID, "class_mapping": CLASS_MAPPING, "positive_class": POSITIVE_CLASS,
    "input_shape": [1280, 16, 16],
    "architecture": {"stem": "Conv1x1 1280->256", "stages": [{"channels": 256, "blocks": 3, "spatial_size": [16, 16]}, {"channels": 384, "blocks": 3, "spatial_size": [8, 8]}, {"channels": 512, "blocks": 2, "spatial_size": [4, 4]}], "mlp_ratio": 3, "dropout": DROPOUT, "stochastic_depth": 0.10},
    "normalization": {"path": str(normalization_path.relative_to(PROJECT_ROOT)), "epsilon": EPSILON, "statistics_split_hash": str(normalization["split_hash"])},
    "split_hash": SPLIT_HASH, "threshold": threshold,
    "training": {"seed": SEED, "batch_size": BATCH_SIZE, "effective_batch_size": BATCH_SIZE * ACCUMULATION_STEPS, "accumulation_steps": ACCUMULATION_STEPS, "max_epochs": MAX_EPOCHS, "best_epoch": best_epoch, "patience": PATIENCE, "optimizer": "AdamW", "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS, "training_time_seconds": training_time_seconds, "amp_enabled": AMP_ENABLED},
    "versions": VERSIONS,
}
with (RUN_DIR / "model_config.json").open("w", encoding="utf-8") as handle:
    json.dump(model_config, handle, ensure_ascii=False, indent=2)

cpu_model = ConvNeXtFeatureClassifier().cpu()
cpu_model.load_state_dict(torch.load(RUN_DIR / "best_state_dict.pt", map_location="cpu", weights_only=False)["model_state_dict"])
cpu_model.eval()
scripted_path = RUN_DIR / "best_model_scripted.pt"
example_input = torch.zeros((1, 1280, 16, 16), dtype=torch.float32)
traced_model = torch.jit.trace(cpu_model, example_input, strict=True)
torch.jit.save(traced_model, scripted_path)

state_model = ConvNeXtFeatureClassifier().to(DEVICE)
state_model.load_state_dict(torch.load(RUN_DIR / "best_state_dict.pt", map_location=DEVICE, weights_only=False)["model_state_dict"])
state_model.eval()
scripted_model = torch.jit.load(scripted_path, map_location=DEVICE).eval()
verification_values, _ = next(iter(validation_loader))
verification_values = verification_values[: min(4, len(verification_values))].to(DEVICE)
with torch.no_grad():
    state_logits = state_model(verification_values).float().cpu().numpy()
    scripted_logits = scripted_model(verification_values).float().cpu().numpy()
if not np.allclose(state_logits, scripted_logits, rtol=1e-4, atol=1e-5):
    raise RuntimeError("TorchScript non riproduce i logit del checkpoint")
print("Verifica di ricarica superata:", RUN_DIR)

In [ ]:
from google.colab import files
import shutil

archive = shutil.make_archive(
    base_name=f"/content/convnext_{RUN_ID}",
    format="zip",
    root_dir=RESULTS_DIR,
    base_dir=f"convnext/{RUN_ID}",
)
files.download(archive)

## Output e artefatti prodotti

Ogni esecuzione crea results/convnext seguito dal run_id e salva:

- best_state_dict.pt: pesi scelti dalla balanced accuracy validation;
- last_checkpoint.pt: stato completo per riprendere training, scheduler, optimizer e scaler;
- best_model_scripted.pt: modello TorchScript portabile;
- model_config.json: architettura, preprocessing, split hash, soglia, hardware e versioni;
- history.csv: andamento per epoca aggiornato anche durante il training;
- metrics.csv e predictions.csv: risultati per train, validation e test nel formato comune;
- figures: loss, accuracy, matrici di confusione, ROC, precision-recall e calibrazione.

I file originali non sono mai modificati. In Colab i file grandi nella VM sono una cache temporanea; checkpoint, predizioni e risultati sono sempre persistiti sotto la root del progetto su Drive.